In [0]:
%run ./_utils

### Snapshot finalizer
Runs after all 11 export tasks + smoke_tests succeed.
1. Aggregates per-entity meta from `_meta/{format}/{entity}.json` into one manifest per format at `full/{date}/{format}/manifest.json`.
2. Updates `full/latest.json` (list of available snapshot dates) so the API can advertise availability.
3. Uploads `RELEASE_NOTES.txt` to `full/{date}/RELEASE_NOTES.txt` and `full/RELEASE_NOTES.txt` (latest pointer).
4. Cleans up `_meta/` and `_temp/` working directories.

In [0]:
import json
import os
from datetime import datetime, timedelta, timezone

date_str = get_snapshot_date()
print(f"Snapshot date: {date_str}")

ENTITIES = [
    "works", "authors", "institutions", "sources", "publishers", "funders",
    "topics", "subfields", "fields", "domains",
    "concepts", "keywords", "awards",
    "continents", "countries", "institution-types", "languages",
    "licenses", "sdgs", "source-types", "work-types",
]

FORMATS = ["jsonl", "parquet"]

#### Build per-format combined manifests

In [0]:
for fmt in FORMATS:
    combined_entries = []
    for entity in ENTITIES:
        meta_path = f"{S3_BASE}/{date_str}/_meta/{fmt}/{entity}.json"
        try:
            content = dbutils.fs.head(meta_path, 16 * 1024 * 1024)
            meta = json.loads(content)
        except Exception as e:
            print(f"  WARNING: missing meta for {entity}/{fmt}: {e}")
            meta = {"record_count": 0, "content_length": 0, "files": []}

        combined_entries.append({
            "entity": entity,
            "record_count": meta.get("record_count", 0),
            "content_length": meta.get("content_length", 0),
            "files": meta.get("files", []),
        })

    total_records = sum(e["record_count"] for e in combined_entries)
    total_size = sum(e["content_length"] for e in combined_entries)
    total_files = sum(len(e["files"]) for e in combined_entries)

    combined_manifest = {
        "date": date_str,
        "format": fmt,
        "meta": {
            "record_count": total_records,
            "content_length": total_size,
        },
        "entities": combined_entries,
    }

    combined_path = f"{S3_BASE}/{date_str}/{fmt}/manifest.json"
    dbutils.fs.put(combined_path, json.dumps(combined_manifest, indent=2), overwrite=True)
    print(f"{fmt}: {total_records:,} records, {total_size / (1024**3):.2f} GB "
          f"across {len(ENTITIES)} entities ({total_files} files) → {combined_path}")

#### Update `latest.json` with available snapshot dates

In [0]:
latest_path = f"{S3_BASE}/latest.json"
# Keep ~24 months of snapshot dates (monthly cadence; quarterly drops are
# a subset). 750 days is enough headroom for irregular re-runs.
cutoff = (datetime.now(timezone.utc) - timedelta(days=750)).strftime("%Y-%m-%d")

try:
    existing = json.loads(dbutils.fs.head(latest_path, 65536))
    available_dates = existing.get("available_dates", [])
except Exception:
    available_dates = []

if date_str not in available_dates:
    available_dates.append(date_str)
available_dates = sorted([d for d in available_dates if d >= cutoff], reverse=True)

latest = {"available_dates": available_dates}
dbutils.fs.put(latest_path, json.dumps(latest, indent=2), overwrite=True)
print(f"Updated {latest_path} ({len(available_dates)} dates)")

#### Upload `RELEASE_NOTES.txt`

In [0]:
import requests

RELEASE_NOTES_URL = (
    "https://raw.githubusercontent.com/ourresearch/openalex-walden"
    "/main/notebooks/snapshot/RELEASE_NOTES.txt"
)

resp = requests.get(RELEASE_NOTES_URL, timeout=30)
resp.raise_for_status()
release_notes = resp.text

# Dated copy under the snapshot directory.
dated_path = f"{S3_BASE}/{date_str}/RELEASE_NOTES.txt"
dbutils.fs.put(dated_path, release_notes, overwrite=True)
print(f"Wrote {dated_path}")

# Latest copy at the snapshot root for client-side discoverability.
latest_notes_path = f"{S3_BASE}/RELEASE_NOTES.txt"
dbutils.fs.put(latest_notes_path, release_notes, overwrite=True)
print(f"Wrote {latest_notes_path}")

#### Clean up working directories

In [0]:
for dirname in ("_meta", "_temp"):
    path = f"{S3_BASE}/{date_str}/{dirname}"
    try:
        dbutils.fs.rm(path, recurse=True)
        print(f"Removed {path}")
    except Exception as e:
        print(f"  warning: could not remove {path}: {e}")